In [1]:
import pandas as pd

# Load CSV (Dataset)
repo_df = pd.read_csv('characteristics_repo.csv')
markdown_irt_df = pd.read_csv('characteristics_irts_markdown.csv')
yaml_irt_df = pd.read_csv('characteristics_irts_yaml.csv')

print(repo_df.head())
print(markdown_irt_df.head())
print(yaml_irt_df.head())


                          full_name  is_fork  has_issues           created_at  \
0        josch/cycles_johnson_meyer    False        True  2012-07-04 12:02:27   
1   hantsy/angularjs-cakephp-sample    False        True  2013-11-06 12:56:34   
2          metabase/metabase-deploy    False       False  2015-10-08 00:01:18   
3  sakshamsharma/http-over-protocol    False        True  2016-08-30 10:53:30   
4                  sradley/overflow    False        True  2020-10-27 06:30:05   

                   last_modified            pushed_at main_language  \
0  Sun, 01 Jan 2023 16:35:16 GMT  2018-07-05 03:54:37          Java   
1  Wed, 07 Dec 2022 18:29:56 GMT  2015-10-08 05:00:41           PHP   
2  Sat, 10 Dec 2022 12:07:00 GMT  2022-08-05 22:57:15         Shell   
3  Tue, 15 Nov 2022 20:41:40 GMT  2016-09-25 04:59:55           C++   
4  Thu, 01 Dec 2022 11:13:38 GMT  2022-12-18 03:42:19            Go   

   total_issues_count  open_issues_count  closed_issues_count  ...  \
0               

In [2]:
# Merge Markdown and YAML IRT characteristics based on 'full_name'
irt_combined_df = pd.merge(markdown_irt_df, yaml_irt_df, on='full_name', how='outer', suffixes=('_markdown', '_yaml'))

# Merge the combined IRT characteristics (Markdown + YAML) with repository characteristics
combined_df = pd.merge(repo_df, irt_combined_df, on='full_name', how='left')


print(f"After merging, the combined dataset has {combined_df.shape[0]} rows and {combined_df.shape[1]} columns.")
print(combined_df.head())



After merging, the combined dataset has 1151463 rows and 44 columns.
                          full_name  is_fork  has_issues           created_at  \
0        josch/cycles_johnson_meyer    False        True  2012-07-04 12:02:27   
1   hantsy/angularjs-cakephp-sample    False        True  2013-11-06 12:56:34   
2          metabase/metabase-deploy    False       False  2015-10-08 00:01:18   
3  sakshamsharma/http-over-protocol    False        True  2016-08-30 10:53:30   
4                  sradley/overflow    False        True  2020-10-27 06:30:05   

                   last_modified            pushed_at main_language  \
0  Sun, 01 Jan 2023 16:35:16 GMT  2018-07-05 03:54:37          Java   
1  Wed, 07 Dec 2022 18:29:56 GMT  2015-10-08 05:00:41           PHP   
2  Sat, 10 Dec 2022 12:07:00 GMT  2022-08-05 22:57:15         Shell   
3  Tue, 15 Nov 2022 20:41:40 GMT  2016-09-25 04:59:55           C++   
4  Thu, 01 Dec 2022 11:13:38 GMT  2022-12-18 03:42:19            Go   

   total_issues_c

In [3]:
rows, columns = combined_df.shape

print(f"The combined dataset has {rows} rows and {columns} columns.")

The combined dataset has 1151463 rows and 44 columns.


In [4]:
# Feature: whether the repository has an IRT (either Markdown or YAML)
combined_df['has_irt'] = combined_df['IRT_name_markdown'].notnull() | combined_df['IRT_name_yaml'].notnull()

# Feature: Repository popularity (based on stars)
combined_df['is_popular'] = combined_df['stargazers_count'].apply(lambda x: 1 if x > 100 else 0)

# Feature: Issue activity (open + closed issues from 'total_issues_countv2')
combined_df['issue_activity'] = combined_df['open_issues_countv2'] + combined_df['closed_issues_countv2']

# Feature: Recent activity (if the repository was updated in the last 6 months)
combined_df['recently_updated'] = combined_df['pushed_at'].apply(
    lambda x: 1 if pd.to_datetime(x) > pd.Timestamp.now() - pd.DateOffset(months=6) else 0
)

print(combined_df[['has_irt', 'is_popular', 'issue_activity', 'recently_updated']].head())

   has_irt  is_popular  issue_activity  recently_updated
0    False           0             1.0                 0
1    False           1             8.0                 0
2    False           0            27.0                 0
3    False           1             1.0                 0
4    False           0             2.0                 0


In [5]:
# Feature: Length of the Markdown IRT body (if available)
combined_df['irt_length_markdown'] = combined_df['IRT_raw_markdown'].apply(lambda x: len(str(x)) if pd.notnull(x) else 0)

# Feature: Length of the YAML IRT body (if available)
combined_df['irt_length_yaml'] = combined_df['IRT_raw_yaml'].apply(lambda x: len(str(x)) if pd.notnull(x) else 0)

# Feature: Total length of the IRT (Markdown or YAML)
combined_df['total_irt_length'] = combined_df['irt_length_markdown'] + combined_df['irt_length_yaml']

# Feature: Number of headlines in Markdown IRT (if available)
combined_df['headline_count_markdown'] = combined_df['headlines'].apply(lambda x: len(eval(x)) if pd.notnull(x) else 0)

# Inspect the IRT-related features
print(combined_df[['irt_length_markdown', 'irt_length_yaml', 'total_irt_length', 'headline_count_markdown']].head())

   irt_length_markdown  irt_length_yaml  total_irt_length  \
0                    0                0                 0   
1                    0                0                 0   
2                    0                0                 0   
3                    0                0                 0   
4                    0                0                 0   

   headline_count_markdown  
0                        0  
1                        0  
2                        0  
3                        0  
4                        0  


In [6]:
# Filter rows where either Markdown or YAML IRT exists
filtered_df = combined_df[(combined_df['IRT_name_markdown'].notnull()) | (combined_df['IRT_name_yaml'].notnull())]

# Check the number of rows after filtering
print(f"The filtered dataset has {filtered_df.shape[0]} rows and {filtered_df.shape[1]} columns.")

print(filtered_df.head())


The filtered dataset has 117212 rows and 52 columns.
                full_name  is_fork  has_issues           created_at  \
11  buddyforms/buddyforms    False        True  2016-04-22 10:02:15   
12  buddyforms/buddyforms    False        True  2016-04-22 10:02:15   
17    gardener/etcd-druid    False        True  2019-09-24 08:57:36   
18    gardener/etcd-druid    False        True  2019-09-24 08:57:36   
19    gardener/etcd-druid    False        True  2019-09-24 08:57:36   

                    last_modified            pushed_at main_language  \
11  Mon, 29 Aug 2022 22:12:53 GMT  2022-12-23 20:18:48           PHP   
12  Mon, 29 Aug 2022 22:12:53 GMT  2022-12-23 20:18:48           PHP   
17  Wed, 04 Jan 2023 10:18:32 GMT  2023-01-28 07:33:02            Go   
18  Wed, 04 Jan 2023 10:18:32 GMT  2023-01-28 07:33:02            Go   
19  Wed, 04 Jan 2023 10:18:32 GMT  2023-01-28 07:33:02            Go   

    total_issues_count  open_issues_count  closed_issues_count  ...  \
11              

In [7]:
import pandas as pd
from transformers import RobertaTokenizer

filtered_df = pd.read_csv('filtered_girt_data.csv')

# Filter for Java repositories only
java_df = filtered_df[filtered_df['main_language'] == 'Java']

# Check the number of rows after filtering for Java repositories
print(f"The filtered Java dataset has {java_df.shape[0]} rows.")

The filtered Java dataset has 9143 rows.


## Step 4.1: Tokenizing the Text Data for BERT

In [9]:
from transformers import BertTokenizer

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the IRT body (if available) or YAML body
java_df['tokenized_body'] = java_df.apply(
    lambda row: tokenizer(row['IRT_raw_markdown'] if pd.notnull(row['IRT_raw_markdown']) else row['IRT_raw_yaml'],
                          padding='max_length', truncation=True, max_length=256, return_tensors="pt")['input_ids']
    if pd.notnull(row['IRT_raw_markdown']) or pd.notnull(row['IRT_raw_yaml']) else None, axis=1
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\Samee\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Samee\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

C:\Users\Samee\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\Samee\AppData\Local\Temp\ipykernel_31520\2720676941.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  java_df['tokenized_body'] = java_df.apply(


In [10]:
# Example: Assign a priority label based on issue activity
java_df['priority_label'] = java_df['issue_activity'].apply(lambda x: 1 if x > 50 else 0)

# Verify that the priority label has been assigned
print(java_df[['full_name', 'issue_activity', 'priority_label']].head())


            full_name  issue_activity  priority_label
13   auth0/auth0-java           158.0               1
14   auth0/auth0-java           158.0               1
15   auth0/auth0-java           158.0               1
16  tesshucom/jpsonic           671.0               1
17  tesshucom/jpsonic           671.0               1


C:\Users\Samee\AppData\Local\Temp\ipykernel_31520\2071546512.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  java_df['priority_label'] = java_df['issue_activity'].apply(lambda x: 1 if x > 50 else 0)


In [11]:
import torch

# Check if there are any None or missing values in tokenized inputs or labels
print("Checking for None values in tokenized inputs and labels...")

missing_inputs = java_df['tokenized_body'].isnull().sum()
missing_labels = java_df['priority_label'].isnull().sum()

print(f"Missing tokenized inputs: {missing_inputs}")
print(f"Missing labels: {missing_labels}")

# Remove rows where tokenized_body or priority_label is None
java_df = java_df.dropna(subset=['tokenized_body', 'priority_label'])

# Check the new shape after removing missing values
print(f"Filtered Java dataset shape after removing missing values: {java_df.shape}")


Checking for None values in tokenized inputs and labels...
Missing tokenized inputs: 0
Missing labels: 0
Filtered Java dataset shape after removing missing values: (9143, 57)


In [12]:
from sklearn.model_selection import train_test_split

# Split the Java dataset into training and validation sets (80% training, 20% validation)
train_df, val_df = train_test_split(java_df, test_size=0.2, random_state=42)

# Check the number of rows in each set
print(f"Training set has {train_df.shape[0]} rows.")
print(f"Validation set has {val_df.shape[0]} rows.")


Training set has 7314 rows.
Validation set has 1829 rows.


In [13]:
# Prepare tokenized inputs and labels for the training set
train_inputs = [x.squeeze(0) for x in train_df['tokenized_body'] if x is not None]
train_labels = list(train_df['priority_label'])

# Prepare tokenized inputs and labels for the validation set
val_inputs = [x.squeeze(0) for x in val_df['tokenized_body'] if x is not None]
val_labels = list(val_df['priority_label'])

# Ensure consistency of lengths for both training and validation sets
assert len(train_inputs) == len(train_labels), "Training inputs and labels must be equal in length."
assert len(val_inputs) == len(val_labels), "Validation inputs and labels must be equal in length."


In [14]:
# Define the BugDataset class
class BugDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

# Prepare encodings and labels for training
train_encodings = {
    'input_ids': torch.stack(train_inputs),  # Stack valid inputs only for training
}
train_labels = torch.tensor(train_labels)  # Use valid labels only for training

# Prepare encodings and labels for validation
val_encodings = {
    'input_ids': torch.stack(val_inputs),  # Stack valid inputs only for validation
}
val_labels = torch.tensor(val_labels)  # Use valid labels only for validation

# Create datasets for training and validation
train_dataset = BugDataset(train_encodings, train_labels)
val_dataset = BugDataset(val_encodings, val_labels)

# Check the length of each dataset
print(f"The training dataset contains {len(train_dataset)} samples.")
print(f"The validation dataset contains {len(val_dataset)} samples.")


The training dataset contains 7314 samples.
The validation dataset contains 1829 samples.


# Step 5: Fine-Tuning BERT for Bug Prioritization

In [15]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

# Load the pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    logging_dir='./logs',  # Directory for storing logs
    save_total_limit=2,    # Limit the number of checkpoints to save
)

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,  # Use validation dataset for evaluation during training
)



model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Samee\anaconda3\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [16]:
# Fine-tune the model
trainer.train()

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


Epoch,Training Loss,Validation Loss
1,No log,0.651145
2,0.650300,0.633025
3,0.631800,0.615511


TrainOutput(global_step=1374, training_loss=0.6320031289623987, metrics={'train_runtime': 17524.1384, 'train_samples_per_second': 1.252, 'train_steps_per_second': 0.078, 'total_flos': 2886591388354560.0, 'train_loss': 0.6320031289623987, 'epoch': 3.0})

In [17]:
# Evaluate the model on the validation set
evaluation_results = trainer.evaluate()

# Print evaluation metrics
print("Evaluation results:", evaluation_results)


Evaluation results: {'eval_loss': 0.6155107021331787, 'eval_runtime': 376.3307, 'eval_samples_per_second': 4.86, 'eval_steps_per_second': 0.306, 'epoch': 3.0}


In [18]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Generate predictions on the validation dataset
predictions = trainer.predict(val_dataset)

# Extract the predicted logits
logits = predictions.predictions
predicted_labels = np.argmax(logits, axis=1)

# Extract the true labels
true_labels = val_labels.numpy()

# Calculate accuracy, precision, recall, and F1-score
accuracy = accuracy_score(true_labels, predicted_labels)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='binary')

# Print the detailed metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")


Accuracy: 0.6720
Precision: 0.6949
Recall: 0.8353
F1-score: 0.7586


In [19]:
# Save the fine-tuned model to the specified directory
model.save_pretrained('./bert-prioritization-model')

# Save the tokenizer as well (since you might need it for future predictions)
tokenizer.save_pretrained('./bert-prioritization-model')

print("Model and tokenizer saved successfully.")


Model and tokenizer saved successfully.


In [22]:
from transformers import BertForSequenceClassification, BertTokenizer

# Load the saved model
model = BertForSequenceClassification.from_pretrained('./bert-prioritization-model')

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained('./bert-prioritization-model')

print("Model and tokenizer loaded successfully.")


Model and tokenizer loaded successfully.
